# New CNN - ankle and back data

I have kept the core architecture from the articlespecifically the 4-layer 1D-CNN with Causal Padding and Global Average Pooling. I simply expanded the input layer to accommodate 9 channels (Ankle L/R + Lumbar gyroscopes). This allows for a direct comparison: we can see exactly how much the extra axial (back) context improves the Precision compared to the original single-sensor setup.

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from scipy.signal import butter, filtfilt

# Force CPU to avoid PTX/CUDA errors if necessary
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    """
    Applique un filtre passe-bande de Butterworth.
    - lowcut: 0.5 (Hz)
    - highcut: 20.0 (Hz)
    - fs: 60 (Fréquence d'échantillonnage de FoG-STAR)
    - order: 4 (Ordre du filtre pour une coupure franche)
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    # filtfilt applique le filtre deux fois (avant/arrière) pour éviter le déphasage
    y = filtfilt(b, a, data, axis=0)
    return y
# DATA SEGMENTATION (9 CHANNELS)
def segment_9ch_data(df, subject_list, w=2, o=0.75, Fs=60):
    windowSize = int(w * Fs)
    overlap = int(o * w * Fs)
    slide = windowSize - overlap
    
    # Selecting 3-axis gyro for Ankle L, Ankle R, and Back
    cols = [
        'ankleL_gyro_x', 'ankleL_gyro_y', 'ankleL_gyro_z',
        'ankleR_gyro_x', 'ankleR_gyro_y', 'ankleR_gyro_z',
        'back_gyro_x', 'back_gyro_y', 'back_gyro_z'
    ]
    
    X, y = [], []
    
    for sub_id in subject_list:
        # Isolate one subject at a time
        sub_df = df[df['subjectID'] == sub_id].reset_index(drop=True)
        
        for i in range(0, len(sub_df) - windowSize, slide):
            window = sub_df[cols].iloc[i:i+windowSize].values
            # Local normalization (Zero-mean) as per original script
            window -= np.mean(window, axis=0)
            X.append(window)
            
            # Labeling: Using the 'mode' (most common label) in the window
            # You can change this to a 0.20 threshold for more sensitivity
            label = sub_df['fog'].iloc[i:i+windowSize].mode().values[0]
            y.append(label)
            
    return np.array(X), np.array(y)

2026-03-17 10:13:28.970299: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 10:13:30.947183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-17 10:13:37.638668: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# --- 2. MODEL ARCHITECTURE (ADAPTED FROM ARTICLE) ---
def build_9ch_article_model(window_size=120, num_channels=9):
    model = Sequential([
        Input(shape=(window_size, num_channels)),
        
        # Two consecutive conv layers (16 filters)
        Conv1D(16, kernel_size=9, activation='relu', padding='causal', kernel_regularizer=l2(0.001)),
        Conv1D(16, kernel_size=9, activation='relu', padding='causal', kernel_regularizer=l2(0.001)),
        MaxPooling1D(pool_size=4),
        Dropout(0.4),
        
        # Third conv block (24 filters)
        Conv1D(24, kernel_size=7, activation='relu', padding='causal', kernel_regularizer=l2(0.001)),
        MaxPooling1D(pool_size=3),
        Dropout(0.4),
        
        # Fourth conv block (24 filters)
        Conv1D(24, kernel_size=5, activation='relu', padding='causal', kernel_regularizer=l2(0.001)),
        
        # Global Average Pooling instead of Flatten (reduces overfitting)
        GlobalAveragePooling1D(),
        
        Dense(48, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.4),
        Dense(1, activation='sigmoid')
    ])
    
    # Lower learning rate (0.0001) for stable fusion training
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, 
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc_roc')])
    return model

In [3]:
fusion_model = build_9ch_article_model()
fusion_model.summary()

2026-03-17 10:13:41.967385: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-03-17 10:13:41.967416: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2026-03-17 10:13:41.967423: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-03-17 10:13:41.967428: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-03-17 10:13:41.967432: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: lab-ch-epfl-346514
2026-03-17 10:13:41.967435: I external/local_xla/xla/stream_executor

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 120, 16)        │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 120, 16)        │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 30, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 30, 24)         │         2,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 10, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 10, 24)         │         2,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 24)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 48)             │         1,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,497 (41.00 KB)

 Trainable params: 10,497 (41.00 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# --- 3. MAIN EXECUTION ---
if __name__ == '__main__':
    print("Loading Dataset...")
    df = pd.read_csv('sensor_data.csv') # Ensure this file has subjectID and 9 gyro columns
    df = df.dropna(subset=['ankleL_gyro_x', 'ankleR_gyro_x', 'back_gyro_x'])   
    gyro_cols = [
    'ankleL_gyro_x', 'ankleL_gyro_y', 'ankleL_gyro_z',
    'ankleR_gyro_x', 'ankleR_gyro_y', 'ankleR_gyro_z',
    'back_gyro_x', 'back_gyro_y', 'back_gyro_z'
    ]

    # 3. Appliquer le filtre Butterworth sur les 9 canaux
    print("Filtrage Butterworth (0.5-20Hz) sur les 9 canaux...")
    #df[gyro_cols] = butter_bandpass_filter(df[gyro_cols].values, lowcut=0.5, highcut=20.0, fs=60)
    # SUBJECT-INDEPENDENT SPLIT (Similar to Article 11/5/6 ratio)
    # Adjust these IDs based on your specific CSV
    train_ids = list(range(1, 12))
    val_ids   = list(range(12, 17))
    test_ids  = list(range(17, 23))
    
    print(f"Segmenting: Train({len(train_ids)} subs), Val({len(val_ids)} subs), Test({len(test_ids)} subs)")
    X_train, y_train = segment_9ch_data(df, train_ids)
    X_val, y_val     = segment_9ch_data(df, val_ids)
    X_test, y_test   = segment_9ch_data(df, test_ids)
    
    # SCALE DATA (Essential when mixing ankles and back sensors)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, 9)).reshape(X_train.shape)
    X_val   = scaler.transform(X_val.reshape(-1, 9)).reshape(X_val.shape)
    X_test  = scaler.transform(X_test.reshape(-1, 9)).reshape(X_test.shape)

    # CLASS WEIGHTS (To handle the rarity of FoG episodes)
    # This prevents the model from just guessing "No FoG" to get high accuracy
    counts = np.bincount(y_train)
    weight_for_0 = (1 / counts[0]) * (len(y_train) / 2.0)
    weight_for_1 = (1 / counts[1]) * (len(y_train) / 2.0)
    class_weights = {0: weight_for_0, 1: weight_for_1}

    print("Building and Training Model...")
    model = build_9ch_article_model()
    
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=64,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=1
    )

    # --- 4. FINAL EVALUATION ---
    print("\n--- FINAL TEST RESULTS ON UNSEEN SUBJECTS ---")
    results = model.evaluate(X_test, y_test)
    print(f"Test Loss: {results[0]:.4f}, Test Accuracy: {results[1]:.4f}, Test AUC: {results[2]:.4f}")

    # Save the model for your presentation
    model.save('fog_fusion_model_9ch_2.h5')
    print("Model saved as fog_fusion_model_9ch_2.h5")

Loading Dataset...
Filtrage Butterworth (0.5-20Hz) sur les 9 canaux...
Segmenting: Train(11 subs), Val(5 subs), Test(6 subs)
Building and Training Model...
Epoch 1/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.7264 - auc_roc: 0.7011 - loss: 0.7529 - val_accuracy: 0.8183 - val_auc_roc: 0.4767 - val_loss: 0.6210
Epoch 2/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7127 - auc_roc: 0.7442 - loss: 0.7249 - val_accuracy: 0.7562 - val_auc_roc: 0.5886 - val_loss: 0.5892
Epoch 3/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7172 - auc_roc: 0.7714 - loss: 0.6900 - val_accuracy: 0.7052 - val_auc_roc: 0.6641 - val_loss: 0.5541
Epoch 4/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7286 - auc_roc: 0.7976 - loss: 0.6461 - val_accuracy: 0.6991 - val_auc_roc: 0.7390 - val_loss: 0.5019
Epoch 5/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7259 - auc_roc: 0.8302 - loss: 0.5886 - val_accuracy: 0.6790 - val_auc_roc: 0.8652 - val_loss: 0.4989
Epoch 6/

Test Loss: 0.4915, Test Accuracy: 0.7492, Test AUC: 0.8889
Model saved as fog_fusion_model_9ch_2.h5


In [7]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Get the raw probability predictions (0.0 to 1.0)
y_pred_probs = model.predict(X_test)

# 2. Convert probabilities to binary classes (0 or 1)
# 0.5 is the standard threshold, but you can adjust it later
y_pred_classes = (y_pred_probs > 0.5).astype(int)

# 3. Calculate F1-score
f1 = f1_score(y_test, y_pred_classes)
print(f"\nTarget F1-Score: {f1:.4f}")

# 4. Detailed Report (Precision, Recall for each class)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=['No FoG', 'FoG']))

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

Target F1-Score: 0.6130

Detailed Classification Report:
              precision    recall  f1-score   support

      No FoG       0.96      0.70      0.81      1392
         FoG       0.46      0.91      0.61       390

    accuracy                           0.75      1782
   macro avg       0.71      0.81      0.71      1782
weighted avg       0.85      0.75      0.77      1782



Only the recall for FoG improved

In [11]:
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
results_per_sub = []

print("\n--- ANALYZING PERFORMANCE BY SUBJECT ---")
# test_ids were subjects 17 to 22
for sub_id in test_ids:
    # 1. Isolate data for this specific subject
    # We use the test_df from earlier to get the indices
    indices = df[df['subjectID'] == sub_id].index
    
    # Note: This logic assumes your X_test corresponds to your test_ids order
    # If you segmented them together, we need to segment them individually here:
    X_sub, y_sub = segment_9ch_data(df, [sub_id])
    
    if len(y_sub) == 0: continue
    
    # 2. Scale
    X_sub_scaled = scaler.transform(X_sub.reshape(-1, 9)).reshape(X_sub.shape)
    
    # 3. Predict
    y_prob = model.predict(X_sub_scaled, verbose=0)
    y_pred = (y_prob > 0.5).astype(int)
    
    # 4. Calculate metrics
    sub_recall = recall_score(y_sub, y_pred, zero_division=0)
    sub_precision = precision_score(y_sub, y_pred, zero_division=0)
    
    results_per_sub.append({
        'SubjectID': sub_id,
        'Windows': len(y_sub),
        'FoG_Windows': np.sum(y_sub),
        'Precision': sub_precision,
        'Recall': sub_recall
    })

# Display as a nice table
performance_df = pd.DataFrame(results_per_sub)
print(performance_df.to_string(index=False))


--- ANALYZING PERFORMANCE BY SUBJECT ---
 SubjectID  Windows  FoG_Windows  Precision   Recall
        17       30            0   0.000000 0.000000
        18      457            0   0.000000 0.000000
        19      727          330   0.610338 0.930303
        20      416           34   0.147826 0.500000
        21       62           11   0.500000 0.909091
        22       90           15   0.307692 0.266667


In [15]:
def analyze_errors_9ch_fusion(sensor_data_path, y_true, y_pred, test_ids, step_size=30):
    ACTIVITY_MAP = {
        1: "Walking", 2: "Sit", 3: "Stand", 
        4: "Sit-to-Stand", 5: "Stand-to-Sit", 
        6: "Turn Right", 7: "Turn Left"
    }
    
    # 1. Charger et nettoyer (identique à l'entraînement)
    full_df = pd.read_csv(sensor_data_path)
    cols_9ch = ['ankleL_gyro_x', 'ankleR_gyro_x', 'back_gyro_x']
    clean_df = full_df.dropna(subset=cols_9ch).copy()
    test_df = clean_df[clean_df['subjectID'].isin(test_ids)].reset_index(drop=True)
    
    # On s'assure que y_true et y_pred sont des vecteurs 1D (fenêtres)
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    task_labels_segmented = []
    
    # 2. On boucle UNIQUEMENT sur le nombre de prédictions disponibles
    for i in range(len(y_pred)):
        start = i * step_size
        end = start + 120 
        
        # Sécurité : si on dépasse le test_df
        if end > len(test_df):
            window_activities = test_df['activity'].iloc[start:]
        else:
            window_activities = test_df['activity'].iloc[start:end]
        
        if not window_activities.empty:
            activity_code = window_activities.mode()[0]
            task_labels_segmented.append(ACTIVITY_MAP.get(activity_code, f"Other ({activity_code})"))
        else:
            task_labels_segmented.append("No Data")
    
    task_labels_segmented = np.array(task_labels_segmented)

    # 3. Compilation des résultats (Vérification des longueurs)
    results = []
    # On restreint y_true et y_pred à la longueur des étiquettes de tâches calculées
    y_true = y_true[:len(task_labels_segmented)]
    y_pred = y_pred[:len(task_labels_segmented)]

    for activity_name in np.unique(task_labels_segmented):
        mask = (task_labels_segmented == activity_name)
        
        # On calcule sur les masques de fenêtres
        total_windows = np.sum(mask)
        # FPs : Fenêtres où le modèle voit un gel mais c'est une activité normale
        fps = np.sum((y_pred[mask] == 1) & (y_true[mask] == 0))
        # TPs : Fenêtres où le modèle voit un gel et c'est vrai
        tps = np.sum((y_pred[mask] == 1) & (y_true[mask] == 1))
        # TNs : Fenêtres où le modèle dit "Normal" et c'est vrai
        tns = np.sum((y_pred[mask] == 0) & (y_true[mask] == 0))
        
        precision = tps / (tps + fps) if (tps + fps) > 0 else 0.0
        specificity = tns / (tns + fps) if (tns + fps) > 0 else 0.0
        
        results.append({
            'Activity': activity_name,
            'Total Windows': total_windows,
            'False Positives (FPs)': int(fps), # Forcer en entier
            'Precision': round(precision, 2),
            'Specificity': round(specificity, 2)
        })

    report_df = pd.DataFrame(results).sort_values(by='False Positives (FPs)', ascending=False)
    # Affichage
    print("\n" + "="*70)
    print("      FIXED PER-ACTIVITY PERFORMANCE REPORT")
    print("="*70)
    print(report_df.to_string(index=False))
    return report_df

# --- RUN ANALYSIS ---
# Remplace y_test et y_pred par tes variables réelles du modèle 9-canaux
report = analyze_errors_9ch_fusion('sensor_data.csv', y_test, y_pred_classes, test_ids=[17, 18, 19, 20, 21, 22])


      FIXED PER-ACTIVITY PERFORMANCE REPORT
    Activity  Total Windows  False Positives (FPs)  Precision  Specificity
       Stand            285                    192       0.03         0.31
     Walking            786                     87       0.45         0.87
   Turn Left            457                     66       0.73         0.75
  Turn Right            180                     41       0.66         0.57
         Sit             41                     13       0.00         0.68
Stand-to-Sit             21                      7       0.00         0.67
Sit-to-Stand             12                      0       0.00         1.00
